[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/adairgranados/adair-granados?quickstart=1)


<!-- gen:values.yml#weeks.08.notebook-title|Quest 08: The Boardroom Story -->
# Week 8 Quest: The Boardroom Story
<!-- /gen -->

*The client's analytics team has made four exploratory charts. You turn one of them into a single slide for the CFO.*

💾 **Start your [Prompt Diary](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-prompt-diary) first.** Open `PromptDiary.md` and add a date. Then add the interactions that matter as you work through this. That diary is this [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest)'s record and the source for your [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record).

## Stage 1: Briefing

*(roughly 10 minutes)*

> **From:** A. Partner, [Runnymede Analytics](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-runnymede-analytics)
> **To:** You ([Junior Analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst))
> **Re:** Marlow & Finch: One slide for Friday's board
>
> You know Marlow & Finch from Week 5. They run twelve home-and-gifts stores. Your weekend-matched table shows the promotion uplift was real everywhere but the Midlands. This morning their CFO, **Priya Nair**, called. Finance says the premium your table reported has been **fading**. Trading says the markdowns still pay for themselves. On Friday the board decides whether the weekend promotion programme continues.
>
> Nair's own analytics team sent her **four exploratory charts**. They are in this notebook, exactly as received. Nair gets **one slide**. It must answer one question: **Should we keep weekend promotions?**
>
> Your Week 5 table gives one average for the whole period, and a single average cannot show a fade. Nair wants the picture **over time**, from two years of till data. She wants one chart that the board can decide from and that is true to the data. If the answer is complicated, the slide should show that.
>
> A.P.

**The business question:** Should Marlow & Finch keep weekend promotions?

**The deliverable:** One slide, built from one of the four [exploratory](https://r4ds.had.co.nz/exploratory-data-analysis.html) charts. It's one figure that a board member can quickly read and also trust.

**About the data:** `net_takings` is average daily takings per store (£k), **already net of promotional markdowns**. So you can compare promotion weekends with ordinary weekends directly. Higher net takings on a promotion weekend mean the promotion was successful.

**[Definition of Done](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-definition-of-done)**: Tick these yourself before deciding on the [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge):

- [ ] Every ✏️ prediction cell in [First look](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look) filled in *before* you run its chart
- [ ] The one chart that answers the question chosen, with your reason stated in one sentence
- [ ] Every declutter checklist item verifiably true of your rebuilt figure, including "nothing that carries information was deleted"
- [ ] *([Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver))* The takeaway title states the finding, the annotation explains it, and the palette is Okabe-Ito. The alt text is written, and the 150–300-word [design rationale](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-design-rationale) is drafted
- [ ] Both [Red Flags](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-red-flags) deletions named (what information was lost, in business terms) and restored
- [ ] Every [Debrief](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-debrief) prompt answered, and today's exchanges added to your Prompt Diary
- [ ] A dated entry in your `QuestLog.md` with the badge you are claiming

**Badge criteria this week:**

- 🥉 **[Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze)**: The right chart chosen (one-sentence justification), then rebuilt through every declutter checklist step, with the uncertainty bands kept
- 🥈 **Silver**: Bronze, *plus* annotation, a takeaway title, an Okabe-Ito palette, and alt text. It also needs the 150–300-word design rationale in the format of the [design guide, section 6](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md)
- 🥇 **[Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold)**: Silver, *plus* an interactive hvPlot alternative with a three-sentence argument for when it beats the static slide. It also needs one extension of your own design for the trading directors' working session, with one sentence on what it adds to their <!-- plain-language: everyday decision --> decision

*Rough timing estimates (minutes): Briefing 10, First look 15, Build 40, Red Flags 15, and Debrief 10.*

### The data

Run the cell below first. It generates Marlow & Finch's last two years: Daily net takings and footfall, which weekends ran promotions, and weekly revenue by department.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(5813)  # module seed: Do not change

# ---- Marlow & Finch: Two years of daily trading, Jul 2024 to Jun 2026 ----
dates = pd.date_range("2024-07-01", "2026-06-30", freq="D")
n = len(dates)
day = pd.DataFrame({"date": dates})
day["is_weekend"] = day["date"].dt.dayofweek >= 5

# One promotion decision per weekend (Saturday and Sunday share it):
sat = day["date"].where(day["date"].dt.dayofweek == 5,
                        day["date"] - pd.Timedelta(days=1))
day["weekend_id"] = sat.where(day["is_weekend"])
saturdays = day.loc[day["is_weekend"], "weekend_id"].unique()
promo_weekends = rng.permutation(saturdays)[: len(saturdays) // 2]
day["promo"] = day["weekend_id"].isin(promo_weekends)

# Average daily net takings per store (£k), NET of promotional markdowns.
t = np.arange(n)
season = 6.0 * np.sin(2 * np.pi * t / 365.25 + 1.2)          # trading seasonality
base = np.where(day["is_weekend"], 58.0, 41.0)               # weekends trade higher
fade = np.linspace(1.0, 0.25, n)                             # promotion fatigue
uplift = np.where(day["promo"], 9.5 * fade, 0.0)             # the promo premium
day["net_takings"] = (base + season + uplift + rng.normal(0, 4.0, n)).round(1)
day["footfall"] = (day["net_takings"] * 32 + rng.normal(0, 150, n)).round().astype(int)

# ---- Weekly revenue by department (£k, chain-wide) ----
departments = ["Kitchen", "Living", "Garden", "Gifts", "Stationery", "Café"]
dept_share = np.array([0.30, 0.17, 0.21, 0.14, 0.10, 0.08])
weeks = pd.date_range("2024-07-06", "2026-06-27", freq="W-SAT")
wt = np.arange(len(weeks))
weekly_total = 2450 + 180 * np.sin(2 * np.pi * wt / 52.18 + 1.2) + rng.normal(0, 60, len(weeks))
weekly = pd.DataFrame(
    {"week": np.repeat(weeks, len(departments)),
     "department": np.tile(departments, len(weeks)),
     "revenue": (np.outer(weekly_total, dept_share)
                 * (1 + rng.normal(0, 0.07, (len(weeks), len(departments))))).round(1).ravel()}
)

print(f"Created day: {len(day)} days ({int(day['promo'].sum())} promotion days) "
      f"and weekly: {len(weekly)} department-week rows, in-memory, seed 5813")

## Stage 2: First look

*(roughly 15 minutes)*

The analytics team's [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) drafted the four charts below unsupervised. Your job is **[PREDICT-RUN](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-predict-run)**. Before you run each chart, write one line: What will it show, and can it answer the CFO's question? After all four, pick one.

> ✏️ **Your prediction:** *(one line: What will this chart show, and can it answer the CFO's question? Write it before you run the cell.)*
>
> 

In [ ]:
# Chart A, from the analytics team's Associate: "revenue composition overview"
share = weekly.groupby("department")["revenue"].sum()
fig, ax = plt.subplots(figsize=(6.5, 6.5))
share.plot.pie(ax=ax, autopct="%1.1f%%", startangle=90, ylabel="")
ax.set_title("Revenue Share by Department (Total, 2 Years)")
plt.show()

> ✏️ **Your prediction:** *(one line: What will this chart show, and can it answer the CFO's question? Write it before you run the cell.)*
>
> 

In [ ]:
# Chart B: "revenue over time, full detail"
piv = weekly.pivot(index="week", columns="department", values="revenue")
fig, ax = plt.subplots(figsize=(10, 5))
for d in piv.columns:
    ax.plot(piv.index, piv[d], linewidth=1.2, label=d)
ax.legend(loc="upper right", ncols=2)
ax.grid(True)
ax.set_ylabel("value")
ax.set_title("Weekly Revenue by Department")
plt.show()

> ✏️ **Your prediction:** *(one line: What will this chart show, and can it answer the CFO's question? Write it before you run the cell.)*
>
> 

In [ ]:
# Chart C: "do promotions drive trade? footfall vs takings, weekend days"
wknd = day[day["is_weekend"]]
fig, ax = plt.subplots(figsize=(8, 5.5))
for flag, colour, label in [(True, "green", "Promo weekend"),
                            (False, "red", "Ordinary weekend")]:
    sub = wknd[wknd["promo"] == flag]
    ax.scatter(sub["footfall"], sub["net_takings"], s=14, alpha=0.5,
               color=colour, label=label)
ax.legend()
ax.grid(True)
ax.set_xlabel("footfall")
ax.set_ylabel("net_takings")
ax.set_title("Footfall vs Net Takings (Weekend Days)")
plt.show()

> ✏️ **Your prediction:** *(one line: What will this chart show, and can it answer the CFO's question? Write it before you run the cell.)*
>
> 

In [ ]:
# Chart D: "promo vs non-promo weekends by quarter"
# (sem = standard error of the mean; mean ± 1.96 × sem draws the 95% band)
wk = day[day["is_weekend"]].copy()
wk["quarter"] = wk["date"].dt.to_period("Q").dt.to_timestamp()
q = (wk.groupby(["quarter", "promo"])["net_takings"]
       .agg(mean="mean", sem="sem")
       .reset_index())
promo_line = q[q["promo"]].reset_index(drop=True)
base_line = q[~q["promo"]].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.5, 5))
for line, label in [(promo_line, "promo=True"), (base_line, "promo=False")]:
    ax.plot(line["quarter"], line["mean"], marker="o", linewidth=1.8, label=label)
    ax.fill_between(line["quarter"],
                    line["mean"] - 1.96 * line["sem"],
                    line["mean"] + 1.96 * line["sem"], alpha=0.25)
ax.set_facecolor("#eaeaf2")
ax.grid(True, color="white", linewidth=1.0)
ax.legend(loc="upper right")
ax.set_ylabel("value")
ax.set_title("Mean Weekend Net Takings by Quarter and Promo Status")
plt.show()

**First look findings.** Each chart performs a different analytical task. At most one of those tasks is the CFO's. Answer the questions below in this cell before Build.

1. For each chart A–D: Which analytical task ([design guide, section 2](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md)) does it perform, and what would you use it for?
2. Which chart is closest to the CFO's question (*Should we keep weekend promotions?*), and what is still wrong with it as a slide? A chart that answers the CFO:
   - Compares **promotion weekends against ordinary weekends**
   - Shows the change **over time**
   - Uses **a level of detail the board works with**
   - Shows **how sure we are**

If the shaded 95% bands in Chart D are new to you, read section 5 of [Introduction.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/Introduction.ipynb), under "Showing how sure you are". That section asks you to get a one-sentence explanation of a band from your Associate. Silver and Red Flags use that sentence.

**✍️ My First look answers:**

> A:
>
> B:
>
> C:
>
> D:
>
> Closest to the CFO's question, and why:

## Stage 3: Build

*(roughly 40 minutes)*

Work **with your Associate** throughout. Use **[GENERATE-JUDGE](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-generate-judge)**: Give it one instruction at a time. The [design guide](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), sections 3–6, is your reference.

### 🥉 Bronze

**Deliverable:** The one chart that answers the CFO's question, chosen and named in one sentence, then rebuilt through the declutter checklist below. The shaded bands are drawn with matplotlib's [`fill_between`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html), the same call the exploratory chart used. Keep them in your rebuild.

**Cleaning:** Use this declutter checklist. Verify each item on your own figure:

- [ ] **1. Spines**: Top and right removed
- [ ] **2. Gridlines**: None, or a few faint horizontal lines only
- [ ] **3. Background**: White, with no tinted panel
- [ ] **4. Legend → direct labels**: Legend box gone, and each line named at its end (*replaced*, never deleted). Each label's colour passes the text contrast test in the [design guide, section 3](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md), step 4
- [ ] **5. Rounding and axis density**: The y-axis label names what it measures and its units ("Mean weekend net takings per store (£k)"), with rounded, sparse ticks
- [ ] **6. Emphasis**: The story series in one strong colour, context in grey, **and nothing that carries information deleted: The 95% bands stay**

Suggested opening prompt to your Associate:

> *"Here is matplotlib code for a quarterly comparison chart [paste your chosen First look cell]. Rebuild it applying, one at a time: Remove top and right spines; …"*

Then give it the remaining steps one at a time, and check each.

In [ ]:
# 🥉 Bronze. My choice, in one sentence (complete the comment, then build):
# The chart that answers the CFO's question is Chart __ because ________________.
#
# Rebuild it below with your Associate, working through the declutter checklist.
# The aggregated quarterly data is already in promo_line / base_line (from First look).

fig, ax = plt.subplots(figsize=(9.5, 5))
# TODO 1: Plot both quarterly mean lines AND keep both 95% bands (fill_between)
# TODO 2: Spines (remove top and right)
# TODO 3: Gridlines (a few faint horizontal lines only)
# TODO 4: Background (white; drop the tinted panel)
# TODO 5: Legend → direct labels at the right-hand end of each line
# TODO 6: Y-axis label with units; rounded ticks
# TODO 7: Emphasis (story colour vs grey context; bands in matching light tints)

pass

### 🥈 Silver

Silver adds the following to your Bronze figure:

1. **Takeaway title**: The finding, in one sentence a board member could repeat from memory. Keep "Mean Weekend Net Takings by Quarter and Promo Status" as the descriptive subtitle. Look at the gap between the two lines on your Bronze figure first. State in the title only what the figure shows, even if Trading hopes for a different answer
2. **Annotation layer**: Add these to the figure
   - Name each line in words
   - Put the size of the promotion premium next to the gap it describes, early and late
   - Mark on the figure what the bands show about how sure we are
3. **Accessible palette**: Use an Okabe-Ito accent with grey context, do not rely on colour alone, and put a source line on the figure. Write the **alt text** (the chart type, the variables, and the takeaway)
4. **The design rationale**: 150–300 words in the Markdown cell below, following the template in the [design guide, section 6](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/dataviz-design-guide.md). Write it *while* you design

*In the workshop, aim to finish Bronze and the Silver **figure**. You can finish the rationale in self-study time, before you open the Debrief.*

In [ ]:
# 🥈 Silver. The boardroom slide: Annotation + takeaway title + accessible palette.
# Build on your Bronze figure, directing your Associate and checking every addition.

# TODO: Takeaway title (finding) + descriptive subtitle
# TODO: Annotations. Name the lines in words; size of the premium, early vs late;
#       what the bands show about how sure we are in the recent quarters
# TODO: Okabe-Ito accent (#0072B2 is a safe choice) + grey context + source line

alt_text = "TODO: chart type + variables + takeaway, one sentence."

pass

**✍️ My design rationale** *(150–300 words: The parts, in order)*:

> **1. The question.** *(What must the board be able to see or decide?)*
>
> **2. Why this form.** *(Which analytical task, why this chart type serves it, why the quarterly level of detail, and why the bands are on the figure.)*
>
> **3. Alternatives rejected.** *(At least one of Charts A–C, or another form you considered, and the specific reason it lost.)*
>
> **4. AI draft vs my changes.** *(What the exploratory Chart D did, what you changed, and why each change mattered. Cite today's Prompt Diary entry.)*

### 🥇 Gold

The Partner, reading over your shoulder: *"Good slide. Nair also mentioned a working session with her trading directors the week after the board; they'll want to interact with it."*

**Part one: The interactive alternative.** Build an **interactive [hvPlot](https://hvplot.holoviz.org/) alternative** to your slide. Use the same `.plot`-style call you met in [Demonstration.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/Demonstration.ipynb), section 5. In the Markdown cell below, add a **three-sentence argument** for when the interactive version beats the static one.

**Part two: An extension of your own.** Build **one thing of your own design** that the trading directors' working session would need. Some ideas:

- The daily weekend points linked under the quarterly means (`day` is still in memory)
- Hover text showing exact quarterly means and confidence bands
- A widget that slices by department or year
- A figure for a fair test, which shows what a month with no promotions would need to look like to settle the question

Whatever you build, state **in one sentence what your extension adds to the directors' <!-- plain-language: everyday decision --> decision**.

In [ ]:
# 🥇 Gold: The interactive alternative, plus an extension of your own design.
# import hvplot.pandas  # noqa: F401
#
# TODO (part one): An hvPlot version of the promo-vs-ordinary story for the
#       working session, e.g. quarterly means with hover, toggleable series.
# TODO (part two): Your own extension, and a comment stating, in one
#       sentence, what it adds to the directors' decision.
# Keep the static slide as the board deliverable; these are for exploration.

pass

**✍️ When does the interactive beat the static?** *(exactly three sentences)*

> 

## Stage 4: Red Flags

*(roughly 15 minutes)*

**[BUG-HUNT](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bug-hunt).** While you were building, the analytics team's Associate rebuilt Chart D. It claims it decluttered the chart for the board. It runs without an error and it *looks* clean. Run it, then audit it against your own Bronze figure.

In [ ]:
# The Associate's 'decluttered' slide. Check whether it is fit for the board.
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.plot(promo_line["quarter"], promo_line["mean"], marker="o", linewidth=2.2)
ax.plot(base_line["quarter"], base_line["mean"], marker="o", linewidth=2.2)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e0e0e0", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylabel("Mean weekend net takings per store (£k)")
ax.set_title("Weekend promotions lift takings: keep them", loc="left", fontweight="bold")
# Decluttered per checklist: Removed boxed legend, removed shaded bands ✨
plt.show()

**Your audit:**

1. **Find it.** The Associate deleted two things, and its comment names both. Which checklist rule did each deletion break?
2. **Explain, in business terms, what information was lost**, in the cell below: What each loss does to the board's <!-- plain-language: everyday decision --> decision. Answer these questions:
   - With no legend and no direct labels, what can a board member conclude from two unnamed lines?
   - What did the bands show about the last two or three quarters that the bare lines and the confident title do not admit?
   - Looking at your Bronze figure: How far apart are the lines lately, compared with the width of the bands?
3. **Fix it by directing your Associate.** Tell it what was lost and why it matters, and have *it* restore both. For example: *"Your declutter deleted the series naming and the uncertainty. Restore the 95% bands, replace the legend with direct end-labels, and change the title to what the last four quarters actually support."* Paste the corrected figure into the fix cell

**✍️ What was lost, in business terms:** *(3–4 sentences: The two deletions, and what each one does to Friday's <!-- plain-language: everyday decision --> decision)*

> 

In [ ]:
# 🚩 Fixed slide, corrected by directing your Associate
# TODO: Paste the corrected figure here and run it.

pass

## Stage 5: Debrief

*(roughly 10 minutes)*

**[TEACH-BACK](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-teach-back).** Answer every prompt in the cells below. Then copy them into your `QuestLog.md`, dated, with the badge you are claiming. Check that today's exchanges are in your Prompt Diary.

**1. Client explanation:** "Explain this week's key technique to the client in three sentences, no jargon." *(For Quest 08: What does 'decision-ready' mean, and why did the board get a single slide?)*

> 

**2. Associate audit:** "Where was your Associate wrong, misleading, or unhelpful this week? Paste the exchange." *(A declutter instruction it over-applied, a palette it wrongly called colour-blind-safe, a title that overclaimed: Anything from today's session.)*

> 

**3. Quiz me:** "Ask your Associate to set you three questions on this week's material. Paste the quiz and answer it *without* AI help." *(Suggested prompt: "Set me three short questions (easy, medium, hard) on takeaway titles, the declutter sequence, and Truth! axes. Don't show answers until I've tried.")*

> 

## 🏁 Sign-off

Before you close, use **Run All**. The notebook should run top to bottom without errors (the TODO cells only `pass`). Then work through the Definition of Done in Stage 1 and tick every item.

Open Debrief.ipynb only once you are done. It contains the model slide, a model rationale to mark yours against, and this week's [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery). The [Week 8 README](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/08/README.md) lists what to do before Week 9's [Gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gallery).